
# FITS instrumental calibration: bias (zero) + dark + flat

Notebook for calibrating a single observing night using `astropy` + `numpy`.

Pipeline:

1. scan FITS headers and classify frames,
2. build a sigma-clipped median **master bias**,
3. build a **master dark-current rate** in ADU/s from bias-corrected darks,
4. build a normalized **master flat for each filter** after bias and dark correction,
5. calibrate science frames:

\[
I_{\rm cal} = \frac{I_{\rm raw} - B - t\,D}{F}
\]

where \(B\) is the master bias, \(D\) is the dark-current image in ADU/s, \(t\) is exposure time, and \(F\) is the normalized master flat.

Edit the configuration cell first so that the FITS keywords match your observatory.


In [1]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import sigma_clip


In [ ]:
# wczytujemy pliki
fits_list = Path(".").glob("*.fits")

# sprawdzamy fitsy
hdul = fits.open(f_name)
for h in hdul:
    print(h.info)


image = hdul[0].data
hdr = hdul[0].header
hdul.close()

# sprawdzamy header
print(hdr)

# sprawdzamy image
print(image.shape)

vmin = np.median(image)  - 1 * np.std(image)
vmax = np.median(image)  + 1 * np.std(image)

plt.figure(figsize=(8, 8))
plt.imshow(image,vmin=vmin,vmax=vmax,cmap='binary')
plt.show()

# lista bias
bias_files = []
if hdr["IMAGETYP"].strip().upper() == "BIAS":
    bias_files.append(f_name)

bias_data = []
for b in bias_files:
    hdul = fits.open(b)
    image = hdul[0].data
    hdul.close()
    bias_data.append(image)

bias_stack = np.array(bias_data)
master_bias = np.median(bias_stack, axis=0)
